# Глава 9. Наивный байесовский классификатор

Код к главе 9 книги 1: формула Байеса, гауссовский наивный байес, спам-фильтр вручную
и в scikit-learn (листинг 9.1<!-- lst:Мультиномиальный -->), сглаживание Лапласа, мультиномиальный и бернуллиевский
варианты, сравнение с логистической регрессией.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markpolyak/ml-textbook-code/blob/main/book1/ch09_naive_bayes.ipynb)

In [ ]:
# В Colab ставим пакет mlbook с данными книги (локально: pip install -e . в корне репозитория)
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/markpolyak/ml-textbook-code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm

from mlbook import examples

## Формула Байеса

Спам — 30 % писем, слово «выигрыш» есть в 40 % спама и в 2 % обычных писем.

In [ ]:
def posterior(prior, like_pos, like_neg):
    return prior * like_pos / (prior * like_pos + (1 - prior) * like_neg)


print(round(posterior(0.3, 0.4, 0.02), 2))

## Гауссовский наивный байес на одном признаке

Класс $-1$: 60 %, $N(2, 1)$; класс $+1$: 40 %, $N(5, 2^2)$. Сравниваются произведения
$P(y)\,p(x \mid y)$; точки пересечения — границы между классами (рис. 9.1, а<!-- fig:nb -->).

In [ ]:
g_neg = lambda x: 0.6 * norm.pdf(x, 2, 1)
g_pos = lambda x: 0.4 * norm.pdf(x, 5, 2)
print(f"x = 3: {g_neg(3):.3f} и {g_pos(3):.3f}, P(+1 | x) = {g_pos(3) / (g_neg(3) + g_pos(3)):.2f}")
diff = lambda x: g_pos(x) - g_neg(x)
print("границы:", round(brentq(diff, 2, 5), 2), "и", round(brentq(diff, -5, 2), 2))

In [ ]:
xs = np.linspace(-1, 11, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(xs, g_neg(xs), label=r"$P(-1)\,p(x\,|\,-1)$")
ax.plot(xs, g_pos(xs), "--", label=r"$P(+1)\,p(x\,|\,+1)$")
ax.fill_between(xs, 0, np.maximum(g_neg(xs), g_pos(xs)), where=g_pos(xs) > g_neg(xs),
                color="#dbe6f4")
ax.set_xlabel("$x$")
ax.legend()
plt.show()

## Спам-фильтр вручную

Шесть писем (табл. 9.1<!-- tbl:spam -->), словарь — четыре слова. Мешок слов строит `CountVectorizer`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texts, labels = examples.spam6()
labels = np.array(labels)
vec = CountVectorizer()
counts = vec.fit_transform(texts).toarray()
words = vec.get_feature_names_out()
print(words)
print(counts)

Частоты слов в классах: $q_{jy} = (N_{jy} + \alpha)/(N_y + \alpha n)$. Без сглаживания
($\alpha = 0$) у слов, не встретившихся в классе, вероятность нулевая.

In [ ]:
def word_probs(alpha):
    N = np.array([counts[labels == c].sum(axis=0) for c in (1, 0)])   # строки: спам, не спам
    return (N + alpha) / (N.sum(axis=1, keepdims=True) + alpha * len(words))


def p_spam(text, alpha):
    x = vec.transform([text]).toarray()[0]
    q = word_probs(alpha)
    score = 0.5 * np.prod(q ** x, axis=1)                             # P(y) · ∏ q^f
    return score[0] / score.sum()


for alpha in (0, 1):
    print(f"α = {alpha}: q(спам) = {word_probs(alpha)[0].round(3)}, q(не спам) = {word_probs(alpha)[1].round(3)}")
    for text in ("выигрыш встреча", "бесплатно отчёт"):
        print(f"   «{text}»: P(спам) = {p_spam(text, alpha):.2f}")

Листинг 9.1<!-- lst:Мультиномиальный -->: то же в scikit-learn. Вероятность спама для письма «выигрыш встреча» —
$8/11 \approx 0{,}73$.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

texts = ["выигрыш выигрыш бесплатно", "бесплатно выигрыш",
         "бесплатно встреча", "отчёт встреча",
         "отчёт отчёт бесплатно", "встреча отчёт"]
y = [1, 1, 1, 0, 0, 0]                      # 1 — спам
model = make_pipeline(CountVectorizer(),
                      MultinomialNB(alpha=1.0))
model.fit(texts, y)
model.predict_proba(["выигрыш встреча"])    # [[0.27, 0.73]]

## Бернуллиевский вариант

Учитывается только наличие слова — и отсутствие тоже. Для письма «встреча» варианты
расходятся: 0,40 у мультиномиального и $4/13 \approx 0{,}31$ у бернуллиевского.

In [ ]:
from sklearn.naive_bayes import BernoulliNB

bern = make_pipeline(CountVectorizer(binary=True), BernoulliNB(alpha=1.0)).fit(texts, y)
for text in ("выигрыш встреча", "встреча"):
    print(f"«{text}»: мультиномиальный {model.predict_proba([text])[0, 1]:.2f},",
          f"бернуллиевский {bern.predict_proba([text])[0, 1]:.2f}")

## Наивный байес — линейная модель

Логит мультиномиального наивного байеса: $\theta_0 = \ln(P(+1)/P(-1))$,
$\theta_j = \ln(q_{j,+1}/q_{j,-1})$. В scikit-learn логарифмы вероятностей слов хранятся
в `feature_log_prob_` (строки — классы 0 и 1).

In [ ]:
nb = model[-1]
theta = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
print(dict(zip(words, theta.round(2))), " θ0 =", nb.class_log_prior_[1] - nb.class_log_prior_[0])
h = model[0].transform(["выигрыш встреча"]).toarray()[0] @ theta
print(f"h = {h:.2f}, σ(h) = {1 / (1 + np.exp(-h)):.2f}")

**Двойной учёт.** Пусть у слова «выигрыш» есть двойник, который всегда стоит рядом с ним.
Наивный байес сложит оба вклада, хотя информации не прибавилось: при прежних весах
$h = 2 \cdot 1{,}39 - 0{,}41$.

In [ ]:
w = dict(zip(words, theta))
h2 = 2 * w["выигрыш"] + w["встреча"]
print(f"h = {h2:.2f}, σ(h) = {1 / (1 + np.exp(-h2)):.2f}")

Если заново обучить модель на письмах с двойником («выигрыш приз»), вероятность
получится немного другой: в спаме стало больше слов, в словаре — на одно слово больше,
и сглаживание меняет все частоты. Но рост от двойного учёта виден и здесь.

In [ ]:
twin = [t.replace("выигрыш", "выигрыш приз") for t in texts]
m2 = make_pipeline(CountVectorizer(), MultinomialNB(alpha=1.0)).fit(twin, y)
print(round(m2.predict_proba(["выигрыш приз встреча"])[0, 1], 2))

## Коррелированные признаки

Два признака с корреляцией 0,8 в обоих классах (рис. 9.1, б<!-- fig:nb -->): наивный байес считает их
независимыми доводами и проводит границу поперёк облака точек, логистическая регрессия
учитывает зависимость. Качество сравним на 100 000 новых объектах.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

S = [[1, 0.8], [0.8, 1]]
rng = np.random.default_rng(3)
A = rng.multivariate_normal([0, 0], S, 60)
B = rng.multivariate_normal([1.5, 0.5], S, 60)
X, yc = np.vstack([A, B]), np.r_[np.zeros(60), np.ones(60)]
gnb = GaussianNB().fit(X, yc)
lr = LogisticRegression(C=100).fit(X, yc)
print("веса логистической регрессии:", lr.coef_[0].round(1))

big = np.random.default_rng(0)
Xn = np.vstack([big.multivariate_normal([0, 0], S, 50_000),
                big.multivariate_normal([1.5, 0.5], S, 50_000)])
yn = np.r_[np.zeros(50_000), np.ones(50_000)]
print(f"доля правильных ответов: наивный байес {gnb.score(Xn, yn):.2f},",
      f"логистическая регрессия {lr.score(Xn, yn):.2f}")

In [ ]:
u, v = np.meshgrid(np.linspace(-3, 4.5, 300), np.linspace(-3, 4.5, 300))
G = np.c_[u.ravel(), v.ravel()]
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(*A.T, facecolors="white", edgecolors="black", s=12)
ax.scatter(*B.T, color="black", s=12)
ax.contour(u, v, gnb.predict_proba(G)[:, 1].reshape(u.shape), levels=[0.5],
           colors="C1", linestyles="--")
ax.contour(u, v, lr.predict_proba(G)[:, 1].reshape(u.shape), levels=[0.5], colors="C0")
ax.set_title("штриховая — наивный байес, сплошная — логистическая регрессия", fontsize=9)
ax.set_aspect("equal")
plt.show()

## Задачи

**1.** Болезнь у 1 % людей, тест находит её у 90 % больных и ошибается у 5 % здоровых.

In [ ]:
print(round(posterior(0.01, 0.9, 0.05), 2))

**2.** Письмо «бесплатно бесплатно встреча».

In [ ]:
print(round(model.predict_proba(["бесплатно бесплатно встреча"])[0, 1], 2))

**3.** Граница при равных дисперсиях: средние 0 и 4, $P(-1) = 0{,}8$, $P(+1) = 0{,}2$.

In [ ]:
x_b = brentq(lambda x: 0.2 * norm.pdf(x, 4, 1) - 0.8 * norm.pdf(x, 0, 1), 0, 4)
print(round(x_b, 2), round(2 + np.log(4) / 4, 2))